# Chapter 1 — Notebook 3: Invariant mass and 4-vectors

**Goals**

- Build Lorentz 4-vectors with the `vector` library via the `topmass.kinematics` helpers.
- Compute invariant masses of multi-object systems.
- See the hadronic-W mass peak emerge from the two leading light jets.

In [ ]:
%matplotlib inline
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

from topmass import io, kinematics, selection, plotting, fitting, neutrino, pairing, weights, style
from topmass.constants import M_W, M_TOP

In [ ]:
io.setup()                                  # select release 2025e-13tev-beta
samples = io.build_samples()                # skim '3J1LMET30', https
events = io.load_process('ttbar', samples, fraction=0.1)
print('Number of events:', len(events))

## Worked example: di-jet mass of the two leading non-b-tagged jets

The hadronic W decays to two light quarks. Selecting the two leading jets that are **not** 
b-tagged and computing their invariant mass should reveal a peak near $m_W = 80.4$ GeV.

In [ ]:
cuts = selection.SemilepCuts()
jets = kinematics.jet_vectors(events)
is_light = events.jet_btag_quantile < cuts.btag_quantile_min
light = jets[is_light]
light = light[ak.num(light) >= 2]
m_jj = (light[:, 0] + light[:, 1]).mass

plt.hist(ak.to_numpy(m_jj), bins=80, range=(0, 200))
plt.axvline(M_W, color='red', label=f'PDG $m_W$ = {M_W:.1f} GeV')
plt.xlabel(r'$m_{jj}$ (two leading light jets) [GeV]'); plt.legend()

## ✏️ Your turn 3.1

▶️ Change the histogram range and re-run.

This builds the **transverse mass** of the leptonic W,
$m_T = \sqrt{2\,p_T^\ell\,E_T^{miss}\,(1 - \cos\Delta\phi)}$, from the leading lepton and the MET.
At what mass does the sharp **edge** sit, and why is it there?

> **Challenge (optional):** split into electrons and muons with `events.lep_type[:, 0]`
> (11 = e, 13 = μ) and overlay the two distributions.

In [ ]:
MT_RANGE = (0, 150)    # ✏️ try (0, 200) — where does the edge sit?

lead_lep = kinematics.leading_lepton(events)
dphi = lead_lep.phi - events.met_phi
mt = np.sqrt(2 * lead_lep.pt * events.met * (1 - np.cos(dphi)))

plt.hist(ak.to_numpy(mt), bins=60, range=MT_RANGE)
plt.axvline(M_W, color='red', label=f'PDG $m_W$ = {M_W:.1f} GeV')
plt.xlabel(r'$m_T(\ell,\ \mathrm{MET})$ [GeV]'); plt.ylabel('Events'); plt.legend()

## ✏️ Your turn 3.2

▶️ Change the histogram range and re-run.

This takes the two leading light (non-b-tagged) jets and the leading b-tagged jet and builds a
first, naive hadronic-top candidate mass $m(jjb)$. A broad bump should appear near the top mass.
(Chapter 3 does this properly with the correct jet pairing.)

> **Challenge (optional):** change `BTAG_MIN` (b-tag tightness) and watch the bump sharpen or wash out.

In [ ]:
MJJB_RANGE = (100, 400)   # ✏️ try (100, 350)
BTAG_MIN   = 4            # ✏️ challenge: try 3, 4, 5 (b-tag tightness)

jets = kinematics.jet_vectors(events)
is_b = events.jet_btag_quantile >= BTAG_MIN
light, bjets = jets[~is_b], jets[is_b]
keep = (ak.num(light) >= 2) & (ak.num(bjets) >= 1)
light, bjets = light[keep], bjets[keep]
m_jjb = (light[:, 0] + light[:, 1] + bjets[:, 0]).mass

plt.hist(ak.to_numpy(m_jjb), bins=60, range=MJJB_RANGE)
plt.axvline(172.5, color='red', label='generator $m_t$ = 172.5 GeV')
plt.xlabel(r'$m(jjb)$ (naive) [GeV]'); plt.ylabel('Events'); plt.legend()